In [ ]:
# transferable code functions
import numpy as np

def padding(layer:np.ndarray, mode:str = 'zero'):


    padded_ly_size = (layer.shape[0]+2, layer.shape[1]+2)
    padded_ly = np.zeros(padded_ly_size)
    padded_ly[1:-1,1:-1] = layer
   
    if mode == 'continue':
        padded_ly[0,1:-1] = layer[0,:]
        padded_ly[-1,1:-1] = layer[-1,:]

        padded_ly[1:-1,0] = layer[:,0]
        padded_ly[1:-1,-1] = layer[:,-1]

        padded_ly[0,0] = layer[0,0]
        padded_ly[0,-1] = layer[0,-1]
        padded_ly[-1,0] = layer[-1,0]
        padded_ly[-1,-1] = layer[-1,-1]

    return(padded_ly)

In [ ]:
import hydra
from omegaconf import DictConfig, OmegaConf
cfg = OmegaConf.load("model.yaml")


In [ ]:
#todo I'd like to create a version of this that can have dynamically sized convolutional layers
import numpy as np
from dataclasses import dataclass




print("start")

@dataclass
class NN_Config:
    input_shape: tuple
    kernel_shape : tuple
    kernel_num: int
    conv_blocks: int
    conv_shape: tuple #this should be replaced by stride eventually?
    output_shape: tuple

    #todo make a more flexible model architechure similar to YOLO


class Layer:
    def __init__(self, ltype:str, index:int,  activations: np.ndarray, z_values: np.ndarray, dims:int, shape: tuple):
        self.activations = activations
        self.z_values = z_values
        self.ltype = ltype
        self.shape = shape
        self.dims = dims
        self.index = index

    @classmethod
    def from_shape(cls, layer_shape, l_type, index, dtype=np.float32, **kwargs):
        activations = np.zeros(layer_shape, dtype=dtype)
        z_vals = np.zeros(layer_shape, dtype=dtype)
        shape = activations.shape
        ltype = l_type
        dims = len(shape)
        index = index
        return cls(ltype, index, activations, z_vals, dims, shape)


class Weights:
    def __init__(self, weights: np.ndarray, index, w_type: str, ly_dim: int, shape: tuple):
        self.weights = weights
        self.shape = shape
        self.w_type = w_type
        self.ly_dim = ly_dim
        self.index = index

    @classmethod
    def filters_from_shape(cls, index, kernel_shape, num_filters, type=np.float32, init=True, **kwargs):
        # replaced the the layer size with the number of filters.  
        # while each filter is applied to every prev_ly location according to the stride, 
        # the filters are unique to themselves, not the a particular layer location
        if init:
        #ly_weights = np.random.rand(prev_ly_size, curr_ly_size)
            ly_weights = np.random.uniform(-1,1, size=(*kernel_shape, num_filters))
        else:
            ly_weights = np.zeros(*kernel_shape, num_filters)

        w_type = "conv"
        ly_dim = len(kernel_shape)
        shape = ly_weights.shape
        index = index
        return cls(ly_weights, index, w_type, ly_dim, shape)

    @classmethod
    def full_connected_from_shape(cls, index, prev_ly_shape: tuple, curr_ly_shape: tuple, dtype=np.float32, init=True, **kwargs):
        if init:
            #ly_weights = np.random.rand(prev_ly_size, curr_ly_size)
            ly_weights = np.random.uniform(-1,1, size=(*prev_ly_shape, *curr_ly_shape))
        else:
            ly_weights = np.zeros(shape=(*prev_ly_shape, *curr_ly_shape))
        w_type = "fc"
        ly_dim = len(curr_ly_shape)
        shape = ly_weights.shape
        index = index
        return cls(ly_weights, index, w_type, ly_dim, shape)

    def from_dynamic_shape(cls, kernel_shape, prev_ly_shape, curr_ly_shape, dtype=np.float32, init=True, **kwargs):
        print("not yet implemented :(")
        print("might be impossible :( :(")


class Biases:
    def __init__(self, biases: np.ndarray, index, dims:int, shape:tuple):
        self.biases = biases
        self.dims = dims
        self.shape = shape
        self.index = index

    @classmethod
    def from_shape(cls, index, curr_ly_shape:tuple, dtype=np.float32, init=True, **kwargs):
        if init:
            biases = np.random.rand(*curr_ly_shape)
        else:
            biases = np.zeros(curr_ly_shape)
        
        dims = len(curr_ly_shape)
        shape = curr_ly_shape
        index = index
        return cls(biases, index, dims, shape)

    def from_dynamic_shape(cls, kernel_shape, prev_ly_shape, curr_ly_shape, dtype=np.float32, init=True, **kwargs):
        print("not yet implemented :(")


class NN:
    def __init__(self, config:DictConfig, layers: list, weights: list, biases: list):
        self.config = config
        self.layers = layers # layers hold both actual activations and z values
        self.weights = weights
        self.biases = biases

    @classmethod
    def create_network(cls, cfg:DictConfig, **kwargs):

        layers = []
        for items in cfg.layers:
            print(cfg.layers[items])
            print(type(cfg.layers[items]))
            layers.append(Layer.from_shape(tuple(cfg.layers[items].shape), cfg.layers[items].type, cfg.layers[items].index))
        

        # layer_fn = Layer.create_layer(arch_config.output_shape)
        # layers = [layer0, *hidden_lys, layer_fn]

        biases =[]
        for items in layers: 
            if items.index != 0:
                biases.append(Biases.from_shape(items.index, items.shape))


        weights = []
        for items in layers: 
            current_ly = items
            if items.index != 0:
                if items.ltype == 'fc':
                    weights.append(Weights.full_connected_from_shape(items.index, prev_ly.shape, items.shape))
                if items.ltype == "conv":
                    weights.append(Weights.filters_from_shape(items.index, tuple(items.weights.kernel_size),  items.weights.filter_num))
            prev_ly = items


        return cls(cfg, layers, weights, biases)

    



def forward(input_vals:np.ndarray, net:NN):
    net.layers[0] = input_vals


    for index in range(1, len(net.layers), 1):
        shape = net.layers[index].activations.shape
        new_z_ly = np.zeros(shape)
        new_ly   = np.zeros(shape)
        #net.z_layers[index] = np.dot(net.layers[(index-1)], net.weights[index-1]) + net.biases[index-1]
        #todo: find a preforment way to do this!
        for i in shape[0]:
            for j in shape[1]:
                new_z_ly[i,j] = net.layers[(index - 1)].activations[i,j]* net.weights[index-1].weights[i,j]
                



#todo:
    #figure out network autocreation
    #figure out padding algorrithm
    #figure out down sizing

SyntaxError: invalid syntax (343568433.py, line 141)

In [25]:
cfg = OmegaConf.load("model.yaml")

model = NN.create_network(cfg)

{'index': 0, 'type': 'input', 'shape': [28, 28], 'weights': None}
<class 'omegaconf.dictconfig.DictConfig'>
{'index': 1, 'type': 'conv2D', 'shape': [28, 28], 'weights': {'kernel_size': [3, 3], 'stride': 1}}
<class 'omegaconf.dictconfig.DictConfig'>
{'index': 2, 'type': 'conv2D', 'shape': [14, 14], 'weights': {'kernel_size': [3, 3], 'stride': 1}}
<class 'omegaconf.dictconfig.DictConfig'>
{'index': 3, 'type': 'conv2D', 'shape': [14, 14], 'weights': {'kernel_size': [3, 3], 'stride': 1}}
<class 'omegaconf.dictconfig.DictConfig'>
{'index': 4, 'type': 'fc', 'shape': [10], 'weights': None}
<class 'omegaconf.dictconfig.DictConfig'>


In [ ]:
item = model.layers
print(item)
print(type(item))

for items in item:
    print(items.shape, type(items.activations), items.index, items.ltype)



In [ ]:
cfg = OmegaConf.load("model.yaml")
print(type(cfg))
print(cfg)
examine_1 = cfg.model.layers.input_ly.shape
print("\nexamine_1: ")
print(examine_1)
print(type(examine_1))
examine_2 = tuple(cfg.model.layers.input_ly.shape)
print("\nexamine_2: ")
print(examine_2)
print(type(examine_2))
#next to figure out how to handle .yaml files and DictConfig files

#net = NN.create_network(cfg)


In [ ]:
import numpy as np
filters  = {"prev_ly_size":28,
            "kernel_size": 3,
            "stride" : 1}
l1 = np.zeros(shape=(filters["prev_ly_size"],))
l11 = np.zeros(shape=(filters["prev_ly_size"],))

l2 = []
l1[0] = 1
for index, values in enumerate(l1):
    if index%filters["stride"] == 0:
        l1[index] = 1


    if l1[index] == 1:
        for x in range(filters["kernel_size"]): 
            y=x+1
            try:
                l11[index + (y - filters["kernel_size"]//2)] = l11[index + (y - filters["kernel_size"]//2)] + 1
            except IndexError as error:
                print("had an index error, continuing")
print(l1,"\n",l11)